This is a simple notebook, build over a day or so to explore the processing of neural calcium channel imagery (something that's now to me).

In [ ]:
from caiman.motion_correction import MotionCorrect
from caiman.source_extraction.cnmf import cnmf, params
from caiman.summary_images import correlation_pnr

import imageio.v3 as iio
from IPython.display import Video

import caiman as cm
import numpy as np
import cv2

In [ ]:
from pathlib import Path

In [ ]:
files = sorted(Path('data/sub-F06/sub-F06_ses-CPFE-F06-S3_ophys').glob('*.avi'))

In [ ]:
fname = files[0]

In [ ]:
m = cm.load(fname)
ms = m.resize(0.5, 0.5, 1)

In [ ]:
lo, hi = np.percentile(ms, [1, 99.5])
u8 = (np.clip((ms - lo) / (hi - lo), 0, 1) * 255).astype(np.uint8)

In [ ]:
b = iio.imwrite("<bytes>", u8, extension=".mp4", fps=30,
                codec="libx264", quality=7, macro_block_size=1)
Video(data=b, embed=True, mimetype="video/mp4")

With these images, we're supposed to see neurons firing, but on the raw adata above, this isn't visible at all. It's more like a mess of haze, with lots of movement, and hard to discern signature, plus a strong constant background. Let's try to fix this, by first removing the low spatial frenquency information.

In [ ]:
low = np.array([cv2.GaussianBlur(frame, (45, 45), 4.5) for frame in ms[4:]])
high = ms[4:] - low

In [ ]:
lo, hi = np.percentile(high, [1, 99.9])
u8h = (np.clip((high - lo) / (hi - lo), 0, 1) * 255).astype(np.uint8)
b = iio.imwrite("<bytes>", u8h, extension=".mp4", fps=30,
                codec="libx264", quality=7, macro_block_size=1)
Video(data=b, embed=True, mimetype="video/mp4")

Low spatial frequency data is removed by subtracting it from the raw data, and the image looks more like a "brain" now, I think I can start to see neurons firing. However, there's a lot of sot noise there.Since this is a quick demo and I don't care about having the best possible extraction, I'll adress it using temporal smoothing

Now to remove the shot noise. I do this with a simple low pass filter in the time domain.

In [ ]:
from scipy import ndimage

In [ ]:
high_slow = ndimage.uniform_filter1d(high, 5, axis=0)

In [ ]:
lo, hi = np.percentile(high_slow, [1, 99.9])
u8s = (np.clip((high_slow - lo) / (hi - lo), 0, 1) * 255).astype(np.uint8)
b = iio.imwrite("<bytes>", u8s, extension=".mp4", fps=30,
                codec="libx264", quality=7, macro_block_size=1)
Video(data=b, embed=True, mimetype="video/mp4")

And here it is, the movie from above, simply averaged in the time axis over five frames, or 167ms at 30fps. Averaging is a bit of a naive way of dealing with shot noise, and using say time correlations of the pixels over time would work better, but then again I just want to see neurons firing now.

There are also a few other things we can start to see: if looks like we have much better resolution and SNR on axis, and that the edges of the field of view are quite distorted, with a "fisheye" effect of some sort. i don't know the exact optical setup, but I'd guess it's a function of the lens that imposes a havy vignetting, and i would exepect a large amout of field curvature as well.

In [ ]:
high_slow_flat = high_slow / np.maximum(low[4:].mean(axis=0)[np.newaxis,:,:], 1)

In [ ]:
lo, hi = np.percentile(high_slow_flat, [1, 99.9])
u8s = (np.clip((high_slow_flat - lo) / (hi - lo), 0, 1) * 255).astype(np.uint8)
b = iio.imwrite("<bytes>", u8s, extension=".mp4", fps=30,
                codec="libx264", quality=7, macro_block_size=1)
Video(data=b, embed=True, mimetype="video/mp4")

In [ ]:
base = ndimage.grey_opening(high_slow, size=(299, 1, 1))   # 299 frames ≈ 10 s, along time only
act = high_slow - base

In [ ]:
lo, hi = np.percentile(act, [1, 99.9])
u8a = (np.clip((act - lo) / (hi - lo), 0, 1) * 255).astype(np.uint8)
ba = iio.imwrite("<bytes>", u8a, extension=".mp4", fps=30,
                codec="libx264", quality=7, macro_block_size=1)
Video(data=ba, embed=True, mimetype="video/mp4")

This is the result of a morphological opening over a ten second window. Anything 

Better, but the little sideways jerks make the image jitter, and that can't be good for a pixel level thresholding based method. Let's go back to the high passed image, and try CaImAn's built in motion correction function.

In [ ]:
mc = MotionCorrect([str(fname)], max_shifts=(35, 35), gSig_filt=(8, 8),
                   pw_rigid=False, border_nan='copy')
mc.motion_correct(save_movie=True)
shifts = np.array(mc.shifts_rig)

In [ ]:
mms = cm.load(mc.mmap_file[0]).resize(0.5, 0.5, 1)

In [ ]:
low_mc = np.array([cv2.GaussianBlur(frame, (45, 45), 4.5) for frame in mms[4:]])
high_mc = mms[4:] - low_mc
high_slow_flat_mc = high_slow / np.maximum(low_mc[4:].mean(axis=0)[np.newaxis,:,:], 1)
high_slow_mc = ndimage.uniform_filter1d(high_mc, 5, axis=0)
base_mc = ndimage.grey_opening(high_slow_mc, size=(299, 1, 1))   # 299 frames ≈ 10 s, along time only
act_mc = high_slow_mc - base_mc

In [ ]:
low_mc = np.array([cv2.GaussianBlur(frame, (45, 45), 4.5) for frame in mms[4:]])
high_mc = mms[4:] - low_mc
high_slow_flat_mc = high_slow / np.maximum(low_mc[4:].mean(axis=0)[np.newaxis,:,:], 1)
high_slow_mc = ndimage.uniform_filter1d(high_mc, 5, axis=0)
base_mc_flip = ndimage.grey_opening(-high_slow_mc, size=(299, 1, 1))   # 299 frames ≈ 10 s, along time only
act_mc_flip = high_slow_mc - base_mc_flip

In [ ]:
# Side by side, same contrast stretch: left = no motion correction, right = rigid NoRMCorre
both = np.concatenate([act, act_mc], axis=2)
lo, hi = np.percentile(both, [1, 99.9])
u8b = (np.clip((both - lo) / (hi - lo), 0, 1) * 255).astype(np.uint8)
bb = iio.imwrite("<bytes>", u8b, extension=".mp4", fps=30, codec="libx264", quality=6, macro_block_size=1)
Video(data=bb, embed=True, mimetype="video/mp4")

In [ ]:
# Side by side, same contrast stretch: left = no motion correction, right = rigid NoRMCorre
both = np.concatenate([act_mc, act_mc_flip], axis=2)
lo, hi = np.percentile(both, [1, 99.9])
u8b = (np.clip((both - lo) / (hi - lo), 0, 1) * 255).astype(np.uint8)
bb = iio.imwrite("<bytes>", u8b, extension=".mp4", fps=30, codec="libx264", quality=6, macro_block_size=1)
Video(data=bb, embed=True, mimetype="video/mp4")

In [ ]:
noise = np.std(act_mc_flip,axis=0)

plt.imshow(noise)

In [ ]:
act_snr = (act_mc / np.maximum(noise, 1e-10)) - 1

act_snr[act_snr < 3] = 0

lo, hi = np.percentile(act_snr, [1, 99.9])
u8a = (np.clip((act_snr - lo) / (hi - lo), 0, 1) * 255).astype(np.uint8)
ba = iio.imwrite("<bytes>", u8a, extension=".mp4", fps=30,
                codec="libx264", quality=7, macro_block_size=1)
Video(data=ba, embed=True, mimetype="video/mp4")

With the motion corrected movie on the right, and the non corected one on the left, we can see the difference it makes. Now let's write a simple, naive pipeline to try to detect activations. We'll define an activation event as time-wise maximum of the activation map:  

In [ ]:
peak = act_snr.max(axis=0)
pk = np.argwhere((peak == ndimage.maximum_filter(peak, size=7)) & (peak > 5))

In [ ]:
from matplotlib import pyplot as plt
plt.imshow(peak)
plt.plot(pk[:,1],pk[:,0],'wx')

Some of these peaks are bunched together, and comparing with the the reference run for this dataset, I can 

Here are the detected peaks, using a simple pileline, and refinding with caiman's centroid finding function. Thsi is as far as I'll take my little bespoke pipeline, because, from reading Caiman's docs, the library can do this quite easily on our little clip, so let's try this.

## CaImAn CNMF-E on the motion-corrected chunk

Same chunk, ×2 downsampled, start-up frames dropped. Parameters are the authors' (stored in the NWB, `processing/ophys/processing_parameters`). Pixel-valued ones are halved for the downsampling: gSig 4 → 2, gSiz 17 → 9, patch half-size rf 40 → 20, patch overlap stride 25 → 12. Ratios are kept (ring_size_factor 1.5). The authors computed the background at 2× spatial downsampling (ssub_B 2), so on data already downsampled ×2 the equivalent is ssub_B 1. I Will not claipm to understand what all the paramters mean, I simply lifted them from the files to try and reproduce their run.

In [ ]:
# CNMF-E wants a C-order memmap; this also downsamples x2 and drops the 4 start-up frames
fname_new = cm.save_memmap([mc.mmap_file[0]], base_name='cnmfe_', order='C',
                           resize_fact=(0.5, 0.5, 1), remove_init=4)
Yr, dims, T = cm.load_memmap(fname_new)
images = np.reshape(Yr.T, [T] + list(dims), order='F')

In [ ]:
opts = params.CNMFParams(params_dict={
    'fr': 29.78, 'decay_time': 1.0, 'p': 1,            # frame rate, GCaMP6s decay (s), AR(1) deconvolution
    'method_init': 'corr_pnr', 'K': None,              # CNMF-E seeding from correlation x PNR, no fixed cell count
    'gSig': (2, 2), 'gSiz': (9, 9), 'center_psf': True,  # cell radius and patch size (px at 300x300)
    'min_corr': 0.9, 'min_pnr': 10,                    # seed thresholds (authors' values)
    'ring_size_factor': 1.5, 'ssub_B': 1,              # ring background: radius = 1.5 x gSiz (authors')
    'nb': 0, 'nb_patch': 0, 'low_rank_background': None,
    'rf': 20, 'stride': 12, 'only_init': True,         # patches: half-size 20 px, overlap 12 px (authors' 40, 25 halved)
    'ssub': 1, 'tsub': 1, 'merge_thr': 0.8,
    'normalize_init': False, 'update_background_components': True,
    'method_deconvolution': 'oasis', 'del_duplicates': True, 'border_pix': 0,
})

In [ ]:
cnm = cnmf.CNMF(n_processes=1, params=opts, dview=None)
cnm.fit(images)          # ~100 s on this chunk (patches run one after another without a cluster)
cnm.estimates.A.shape, cnm.estimates.C.shape   # (pixels, components), (components, frames)

In [ ]:
import matplotlib.pyplot as plt
from caiman.utils.visualization import plot_contours
cn, pnr = correlation_pnr(images, gSig=2, swap_dim=False)
C = cnm.estimates.C[:8]
fig, ax = plt.subplots(1, 2, figsize=(12, 6))
plt.sca(ax[0]); plot_contours(cnm.estimates.A, cn * pnr, display_numbers=False)
ax[1].plot(C.T / C.max(axis=1) + np.arange(len(C)))
ax[1].set_xlabel('frame'); ax[1].set_ylabel('component (normalised C)');

Now the interesting question: how well did my on naive pipeline actually locate activations? Let's plot the countours in white as before, and my own centroids in red.

In [ ]:
from caiman.utils.visualization import get_contours
conts = get_contours(cnm.estimates.A, dims, thr=0.2, thr_method='max')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
for i, c in enumerate(conts):
    ax.plot(*c['coordinates'].T, 'k', lw=0.8, label="CaImAn's detected contours" if i == 0 else None)
ax.plot(pk[:, 1], pk[:, 0], 'rx', label="My own naive pipeline, centroids")
ax.set_xlim(0, dims[1]); ax.set_ylim(dims[0], 0); ax.set_aspect('equal')
ax.legend()

And the comparison. Of course, an afternoon of tinkering on data I hadn't seen before doesn't agree perfectly with CaImAn, a tool years in the making, by domain level experts, but I'm pretty happy with how much it agrees with CNMF-E as specified. For the numbers:

In [ ]:
from scipy.spatial import cKDTree
centres = cm.base.rois.com(cnm.estimates.A, *dims)   # (row, col), same order as pk
d_pk, _ = cKDTree(centres).query(pk)   # each of my peaks -> nearest CaImAn centre
d_cm, _ = cKDTree(pk).query(centres)   # each CaImAn centre -> nearest of my peaks
r = 3   # match radius, px (9.6 um at 3.2 um/px)
print(f"precision: {(d_pk <= r).sum()}/{len(pk)} of my peaks are within {r} px of a CaImAn centre")
print(f"recall:    {(d_cm <= r).sum()}/{len(centres)} CaImAn components have one of my peaks within {r} px")

The output above corresponds to the share of caiman detected peaks I also detect, and to the share of my detection CaImAn also detects, using the criterion of a peak falling whithin 3 pixels of a CaImAn component.

This was from tuning both the threshold and the window size by hand, but no tuning of any other parameters: I originally set them to values that seemed reasonible and didn't touch them.